# Sprint E8 walkthrough: sizing and portfolio construction

In [1]:
# the repository root is importable so the package and the dashboard
# module can be imported without installing the wheel
import json
import sys
from pathlib import Path

import numpy as np
import pandas as pd

ROOT = Path.cwd()
if not (ROOT / "efb").exists():
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
DATA = ROOT / "data"
PORTFOLIOS = DATA / "portfolios"

In [2]:
# the data hash in the results file must be the hash of the artifacts
# the criteria are read from, recomputed now, not copied
from efb import evaluate

stored = json.loads(
    (ROOT / "sprints" / "E8" / "RESULTS.json").read_text()
)
assert evaluate.e8_data_hash(DATA) == stored["data_hash"], "artifact hash drift"
print("data_hash", stored["data_hash"])
print("verdicts:", stored["reference_values"]["verdicts"])

data_hash be7d29e55c4d8b1d6deddc7e570b53a85f06d0fe58a5bcf461ed0d8d094b8074
verdicts: {'F8.1': 'fail', 'F8.2': 'pass', 'F8.3': 'pass', 'F8.4': 'fail', 'F8.5': 'pass', 'F8.6': 'pass'}


## 1. Every criterion, its stored number and its verdict

In [3]:
for name, block in stored["criteria"].items():
    print(name, block["verdict"], json.dumps(block["stored_numbers"])[:160])

F8.1 fail {"max_abs_weight_difference": 0.008462513398020488, "n_dates_compared": 2595}
F8.2 pass {"mean_idio_share_after_fmp": 1.0}
F8.3 pass {"max_violation": 3.954455873866891e-09, "solver_fallbacks": 0, "solver_fallback_rate": 0.0, "fallbacks_by_rho_seed": {}}
F8.4 fail {"0.02": {"dispersion_lambda_0": 1.4163803395852619, "dispersion_after_shrinkage": 1.4163803395852619, "lambda_chosen": 0.0}, "0.05": {"dispersion_lambda_0": 1.
F8.5 pass {"combined": {"0.02": {"realized_ir": 0.5578689092404094, "realized_ic": 0.018931499780912475, "n_eff": 128.70507656549543, "n_names": 456.63428571428574, "pred
F8.6 pass {"combined": {"champion_model": "xs_v1", "per_family_alternative": "xs_v1", "mean_idio_share_after_fmp_champion": 1.0, "mean_idio_share_after_fmp_alternative": 


## 2. The synthetic alpha, by hand

z(i,t) = rho * standardized(e(i,t+h)) + sqrt(1 - rho^2) * eps(i,t). This is a controlled experiment that uses future data deliberately, never a backtest.

In [4]:
from efb import size

frame = size.synthetic_alpha(DATA, rho=0.05, seed=0)
measured = float(frame.groupby("date")["ic"].mean().mean())
print("measured mean IC", round(measured, 4), "against rho 0.05")
assert abs(measured - 0.05) < 0.05
assert (frame["source"] == "synthetic controlled experiment").all()

measured mean IC 0.0486 against rho 0.05


## 3. The transfer coefficient table (F8.5)

The realized IR against IC * sqrt(n_eff); the ratio is the transfer coefficient.

In [5]:
table = stored["criteria"]["F8.5"]["stored_numbers"]
for construction, block in table.items():
    for rho, numbers in block.items():
        print(construction, rho, "IR", round(numbers["realized_ir"], 3),
              "TC_neff", round(numbers["transfer_coefficient_neff"], 3),
              "TC_n", round(numbers["transfer_coefficient_n"], 3))

combined 0.02 IR 0.558 TC_neff 2.597 TC_n 1.379
combined 0.05 IR 0.896 TC_neff 1.728 TC_n 0.917
combined 0.1 IR 0.659 TC_neff 0.644 TC_n 0.342
mv_constrained 0.02 IR 0.278 TC_neff 1.295 TC_n 0.688
mv_constrained 0.05 IR 0.65 TC_neff 1.255 TC_n 0.666
mv_constrained 0.1 IR 1.193 TC_neff 1.165 TC_n 0.618
mv_unconstrained 0.02 IR 0.39 TC_neff 1.815 TC_n 0.964
mv_unconstrained 0.05 IR 0.936 TC_neff 1.806 TC_n 0.959
mv_unconstrained 0.1 IR 1.776 TC_neff 1.734 TC_n 0.921
procedure_6_3 0.02 IR 0.378 TC_neff 1.761 TC_n 0.935
procedure_6_3 0.05 IR 0.905 TC_neff 1.746 TC_n 0.927
procedure_6_3 0.1 IR 1.722 TC_neff 1.682 TC_n 0.893
proportional 0.02 IR 0.399 TC_neff 1.859 TC_n 0.987
proportional 0.05 IR 0.832 TC_neff 1.605 TC_n 0.852
proportional 0.1 IR 0.866 TC_neff 0.846 TC_n 0.449
sharpe 0.02 IR 0.399 TC_neff 1.859 TC_n 0.987
sharpe 0.05 IR 0.832 TC_neff 1.605 TC_n 0.852
sharpe 0.1 IR 0.866 TC_neff 0.846 TC_n 0.449
shrunk 0.02 IR 0.399 TC_neff 1.859 TC_n 0.987
shrunk 0.05 IR 0.832 TC_neff 1.605 

## 4. Robustness and shrinkage (F8.4)

In [6]:
resampling = pd.read_parquet(PORTFOLIOS / "e8_f84_resampling.parquet")
print(resampling.to_string())

    rho  dispersion_lambda_0  dispersion_after_shrinkage  lambda_chosen
0  0.02             1.416380                    1.416380            0.0
1  0.05             1.415278                    1.415278            0.0
2  0.10             1.411274                    1.411274            0.0


## 5. Gate G3: the construction stack runs end to end

In [7]:
weights = pd.read_parquet(PORTFOLIOS / "proportional.parquet")
for column in ("weight", "idio_share", "idio_share_after_fmp",
               "factor_variance", "n_eff"):
    assert column in weights.columns, column
print("proportional book columns", sorted(weights.columns))
print("G3: alpha to hedged, sized book runs end to end")

proportional book columns ['date', 'factor_variance', 'gross', 'idio_share', 'idio_share_after_fmp', 'idio_variance', 'max_violation', 'n_eff', 'net', 'rho', 'seed', 'ticker', 'weight']
G3: alpha to hedged, sized book runs end to end


## 6. The D7 panel map

In [8]:
from dashboard.tabs import d07_sizing as d7

summary = d7.load_summary()
assert not summary.empty
print("D7 reads the construction summary and the weight artifacts")

D7 reads the construction summary and the weight artifacts


## 7. The memo's evidence, in citation order

In [9]:
memo = (ROOT / "docs" / "research" / "E8_construction_memo.md").read_text()
joined = " ".join(memo.split())
for name in ("F8.1", "F8.2", "F8.3", "F8.4", "F8.5", "F8.6"):
    assert name in joined, name
assert "What would falsify this?" in joined
print("memo cites every criterion and the falsification section")

memo cites every criterion and the falsification section


In [10]:
# closing checklist: every criterion name is covered by the code
import json as _json

source = "\n".join(
    "".join(cell["source"])
    for cell in _json.loads(
        (ROOT / "notebooks" / "E8_walkthrough.ipynb").read_text()
    )["cells"]
    if cell["cell_type"] == "code"
)
assert all(
    name in source
    for name in ("F8.1", "F8.2", "F8.3", "F8.4", "F8.5", "F8.6")
)
print("closing checklist: clean")

closing checklist: clean
